# 02 · Preprocessing

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv("../data/application_train.csv")

# Same split as notebook 04. Fill values are learned on the training rows only.
train_idx, test_idx = train_test_split(df.index, test_size=0.2, stratify=df["TARGET"], random_state=42)

# 1. Sentinel values

In [3]:
print("DAYS_EMPLOYED == 365243:", (df["DAYS_EMPLOYED"] == 365243).sum())
df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)

# Other placeholders: positive DAYS_ values and 'XNA' codes
days_cols = [c for c in df.columns if c.startswith("DAYS_")]
print("Positive DAYS_ values:", int((df[days_cols] > 0).sum().sum()))
for col in df.select_dtypes(include="object").columns:
    n = (df[col] == "XNA").sum()
    if n:
        print(f"{col}: {n} rows are 'XNA'")

# Setting `AMT_INCOME_TOTAL` 117,000,000 value to NaN. 
# It changes only that one value, keeps the row, and lets the imputer fill it like any other missing income.
df.loc[df["AMT_INCOME_TOTAL"] > 1e8, "AMT_INCOME_TOTAL"] = np.nan
print("AMT_INCOME_TOTAL max:", df["AMT_INCOME_TOTAL"].max())

DAYS_EMPLOYED == 365243: 55374
Positive DAYS_ values: 0
CODE_GENDER: 4 rows are 'XNA'
ORGANIZATION_TYPE: 55374 rows are 'XNA'
AMT_INCOME_TOTAL max: 18000090.0


- `DAYS_EMPLOYED = 365243` (55,374 pensioners/unemployed) → `NaN`; the missing flag below keeps the "not employed" information.
- `ORGANIZATION_TYPE = XNA` is the same pensioner group, so it stays as its own category.
- `CODE_GENDER = XNA` (4 rows) doesn't matter: gender is not used as a feature (notebook 03).
- `AMT_INCOME_TOTAL` max value (117,000,000) set down to nan so it can be filled up later

# 2. Features with more than 40% missing

In [4]:
missing_pct = df.isnull().mean().sort_values(ascending=False)
print(f"{(missing_pct > 0.4).sum()} columns are more than 40% missing")
(missing_pct[missing_pct > 0.4] * 100).round(1)

49 columns are more than 40% missing


COMMONAREA_AVG                  69.9
COMMONAREA_MODE                 69.9
COMMONAREA_MEDI                 69.9
NONLIVINGAPARTMENTS_AVG         69.4
NONLIVINGAPARTMENTS_MODE        69.4
NONLIVINGAPARTMENTS_MEDI        69.4
FONDKAPREMONT_MODE              68.4
LIVINGAPARTMENTS_MEDI           68.4
LIVINGAPARTMENTS_AVG            68.4
LIVINGAPARTMENTS_MODE           68.4
FLOORSMIN_AVG                   67.8
FLOORSMIN_MODE                  67.8
FLOORSMIN_MEDI                  67.8
YEARS_BUILD_AVG                 66.5
YEARS_BUILD_MEDI                66.5
YEARS_BUILD_MODE                66.5
OWN_CAR_AGE                     66.0
LANDAREA_MEDI                   59.4
LANDAREA_AVG                    59.4
LANDAREA_MODE                   59.4
BASEMENTAREA_AVG                58.5
BASEMENTAREA_MEDI               58.5
BASEMENTAREA_MODE               58.5
EXT_SOURCE_1                    56.4
NONLIVINGAREA_MEDI              55.2
NONLIVINGAREA_AVG               55.2
NONLIVINGAREA_MODE              55.2
E

**Decision: keep them all, impute, and add a missing indicator.**
- **EXT_SOURCE_1** (56%): one of the strongest predictors; dropping it loses signal.
- **OWN_CAR_AGE** (66%): missing just means "no car".
- **Housing block** (`*_AVG / *_MODE / *_MEDI`, 47–70%): weak on their own, but the tree models can ignore them and the missing flags carry information.

# 3. Missing indicators - are they predictive?

In [5]:
missing_cols = missing_pct[missing_pct > 0.05].index
indicators = df[missing_cols].isnull().astype(int).add_suffix("_MISSING")
df = pd.concat([df, indicators], axis=1)

lr = LogisticRegression(max_iter=1000)
lr.fit(indicators.loc[train_idx], df.loc[train_idx, "TARGET"])
auroc = roc_auc_score(df.loc[test_idx, "TARGET"], lr.predict_proba(indicators.loc[test_idx])[:, 1])

print(f"Number of missing indicators: {indicators.shape[1]}")
print(f"AUROC using missing indicators only: {auroc:.4f}")

Number of missing indicators: 58
AUROC using missing indicators only: 0.5858


**AUROC 0.586 from the indicators alone** — better than random (0.5), so *whether* a field is blank is predictive. The indicators stay as features.

# 4. Imputation
- **Median for numeric**: money columns are right-skewed; the median is robust to outliers.
- **Mode for categorical**: text has no median.

Both are computed on the training rows only and applied to all rows.

In [6]:
num_cols = df.select_dtypes(include="number").columns.drop("TARGET")
cat_cols = df.select_dtypes(include="object").columns

fill_values = pd.concat([df.loc[train_idx, num_cols].median(),
                         df.loc[train_idx, cat_cols].mode().iloc[0]])
df = df.fillna(fill_values)
print("Missing values left:", df.isnull().sum().sum())

Missing values left: 0


In [7]:
df.to_csv("../data/cleaned_application_train.csv", index=False)
df.shape

(307511, 180)